# IrriGator - Initialize

First notebook to run, building the **essential blocks** to ensure data ingestion

## 0. Setup

In [ ]:
import logging
import os
from datetime import date, timedelta
from pathlib import Path

# Show info-level logs in the notebook
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(name)s] %(levelname)s: %(message)s",
    datefmt="%H:%M:%S",
)

REPO_ROOT = os.path.dirname(os.getcwd())  # adjust if needed
os.chdir(REPO_ROOT)

# Paths — all relative to REPO_ROOT
RAW_DIR = Path("data/raw")  # ERA5-Land, SEAS5, IFS ENS raw downloads
PROCESSED_DIR = Path("data/processed")  # daily aggregates, cached results
STATIC_DIR = Path("data/static")  # static ERA5-Land data (soil, land cover, etc.)


print(f"Repo root: {REPO_ROOT}")
print(f"Raw dir:   {RAW_DIR}")
print(f"Processed: {PROCESSED_DIR}")

## 1. Generate a region config

Requires ADMIN EXPRESS shapefile from IGN ([Link](https://data.geopf.fr/telechargement/download/ADMIN-EXPRESS-COG-CARTO/ADMIN-EXPRESS-COG-CARTO_3-2__SHP_LAMB93_FXX_2023-05-03/ADMIN-EXPRESS-COG-CARTO_3-2__SHP_LAMB93_FXX_2023-05-03.7z)) - download it and unzip it into `data/static/admin_express` - ideally put only what is inside the ADECOGC_{X-X}_SHP_LAMB93_FXX folder (**or adjust the ADMIN_SHP below**).

Necessary to run this only for a new region. Skip if you already built `configs/{department}.yaml` (e.g. dordogne.yaml)

In [ ]:
from irrigator.region_builder import build_region_config

# Change these for your département
DEPT_CODE = "24"
DEPT_NAME = "dordogne"
OUTPUT_PATH = REPO_ROOT + f"/configs/{DEPT_NAME}.yaml"
ADMIN_SHP = REPO_ROOT + "/data/static/admin_express/DEPARTEMENT.shp"  # adjust if needed

config = build_region_config(
    dept_code=DEPT_CODE,
    output_path=OUTPUT_PATH,
    admin_shp=ADMIN_SHP,
    parent_path=REPO_ROOT,
)

print(f"Region: {config['region']['name']}")
print(f"WGS84 bbox: {config['region']['bbox_wgs84']}")
print(f"L93 bbox:   {config['region']['bbox_l93']}")
print(f"Written to: {OUTPUT_PATH}")


## 2. Load region config & build target grid

In [ ]:
from irrigator.config import load_region_config
from irrigator.ingestion.grid import build_grid

cfg = load_region_config(REPO_ROOT + f"/configs/{DEPT_NAME}.yaml")
grid = build_grid(cfg)

print(f"Region:     {cfg.name} (dept {cfg.code_departement})")
print(f"CRS:        {cfg.crs}")
print(f"Grid shape: {grid.shape[0]} rows × {grid.shape[1]} cols")
print(f"Grid cells: {grid.n_cells:,}")
print(f"Resolution: {grid.resolution_m} m")
print(f"Raw dir:    {cfg.raw_dir}")
print(f"Static dir: {cfg.static_dir}")

## 3. Fetch ERA5-Land data

Necessary to initialize the crop, to have data for current season.

In [ ]:
# France-wide, no per-region dependency
from irrigator.ingestion.cds_client import fetch_era5_land_range

START_DATE = date(2026,1,1)
END_DATE = date(2026,6,1) # can be set to date.today()

fetch_era5_land_range(
    START_DATE, END_DATE, raw_dir=RAW_DIR,
    overwrite=False,
    force_cds_refresh=False # Useful when a downloaded month is incomplete because not yet available
)

## 4. Fetch static layers

#### 4.1 Download the EU-SoilHydroGrids TIFFs in `data/static/dordogne/eu_soilhydrogrids/`: 
--------
1. Register (free) and download from ESDAC:
       https://esdac.jrc.ec.europa.eu/content/3d-soil-hydraulic-database-europe-1-km-and-250-m-resolution
   Extract into ``data/static/eu_soilhydrogrids/``

2. Download the tile navigation GeoPackage from:
       https://github.com/LandscapeGeoinformatics/EU-SoilHydroGrids_tiles_nav
   Place ``grid_cells_250m_wgs84.gpkg`` (or .shp) in the same directory.
   This spatial index tells the loader which tile folders cover your region
   without scanning all 1300 folders.

#### 4.2 Download the DEM tiles in `data/static/dem/`:
--------
RGE ALTI 5 m is free from IGN Géoservices:
    https://geoservices.ign.fr/rgealti

Download the tiles covering Dordogne and place the extracted folder (like `RGEALTI_MNT_5M_ASC_LAMB93_IGN69_D{DEPT_CODE}`):

    data/static/dem/

The loader will mosaic all tiles found in that directory.

#### 4.3 Load both and process them for the specific region

In [ ]:
# --- Build static layers (skip if already done) ---
# These still use RegionConfig because they depend on the regional DEM/soil grids
from irrigator.ingestion.esdac_loader import load_soil_hydro, save_soil_hydro
from irrigator.ingestion.ign_loader import load_terrain, save_terrain
from irrigator.ingestion.grid import build_grid

grid = build_grid(cfg)
soil_grid = load_soil_hydro(cfg, grid)
save_soil_hydro(soil_grid, cfg.processed_dir / "static")
terrain_grid = load_terrain(cfg, grid)
save_terrain(terrain_grid, cfg.processed_dir / "static")
